# 字符串的模式匹配

In [1]:
import re; 

In [2]:
motto = "Verba volant, scripta manent"; 

In [3]:
#构造Unicode从U+0000到U+FFFF的所有字符顺次连接组成的字符串
char = str().join( 
    chr(x) for x in range(65536)
); 

在`ipython`命令行或者`Jupyter notebook`中, 我们可以直接使用特定的转义字符, 改变打印到`sys.stdout`或者笔记本输出单元的字符样式. 本文档据此构造函数, 实现正则表达式匹配内容标记功能. 

In [4]:
#构造生成器, 用于从可迭代对象中有重叠顺次获取相邻两个元素
import copy;  
def adjecent(iter_0): 
    iter_1 = copy.copy(iter_0).__iter__(); 
    iter_2 = copy.copy(iter_0).__iter__(); 
    iter_2.__next__(); 
    for elem_2 in iter_2: 
        elem_1 = iter_1.__next__(); 
        yield elem_1, elem_2; 

In [5]:
#计算正则表达式regex在字符串text中匹配所得子串的起止位置
def pattern_position(text, regex): 
    indices = tuple(
        match.span() for match in re.compile(regex).finditer(text)
    ); 
    return(indices); 

In [6]:
#将正则表达式regex在字符串text中匹配所得子串分布范围使用转义序列标识, 
#以便使用io.TextIOWrapper.write方法打印到stdout或笔记本输出单元
def pattern_highlight(text, regex): 
    indices = pattern_position(text, regex); 
    text_disp = str(); 
    substr_stat = bytearray(len(text) + 1); 
    for idx in indices: 
        start, end = idx; 
        substr_stat[start + 1: end + 1] = (1, ) * (end - start); 
    substr_stat = bytes(substr_stat); 
    for (ch, (start, end)) in zip("\x00" + text, adjecent(substr_stat)): 
        text_disp += ch; 
        if not bool(start) and bool(end): 
            text_disp += "\x1b[07m"; 
        elif bool(start) and not bool(end): 
            text_disp += "\x1b[0m"; 
    text_disp = "\x1b[0m{raw:s}{end:s}\x1b[0m".format(
        raw=text_disp[1:], end=text[-1]
    ); 
    return(text_disp); 

In [7]:
#正则表达式及其匹配结果的对比显示
def pattern_match_illustrate(text, patterns): 
    for regex in [str(), ] + patterns: 
        print("{regex:\x20<24s}{disp:<s}".format(
            regex=regex[: 23], disp=pattern_highlight(text, regex)
        ) )

## 正则表达式语法
注意: 所有的正则表达式语法均不能匹配空子串

### 单字符匹配语法
|模式|功能|备注|
|:-|:-:|:-|
|`a`|字面意义上匹配普通字符`a`|以下字符需要使用反斜杠(`\`)转义: <br>`(`, `)`, `[`, `]`, `\`, `.`, `^`, ` `, <br>`*`, `+`, `?`, `\|`|
|`.`|匹配除`\n`(换行符)以外的<br>任何字符|当启用`re.DOTALL`时, 解除对<br>`\n`的匹配限制|
|`[abc]`|匹配`a`, `b`, `c`之一|在方括号中, `-`  (半角负号) 作为<br>待匹配的字符时, 需要转义为`\-`|
|`[^abc]`|匹配除`a`, `b`, `c`以外的任何字符||
|`[a-z]`|匹配字符编码满足 (大于等于`a`且<br>小于等于`z`) 的字符||
|`[^a-z]`|匹配字符编码不满足 (大于等于`a`<br>且小于等于`z`) 的字符||
|`\w`|匹配`_` (半角下划线) , 或者<br>调用`isalnum`方法返回<br>`True`的单个字符||
|`\W`|匹配调用`isalnum`方法返回<br>`False`的单个字符, 不包括`_`||
|`\d`|匹配调用`isdecimal`方法返回<br>`True`的单个字符||
|`\D`|匹配调用`isdecimal`方法返回<br>`False`的单个字符||

* `[abc]`, `[^abc]`, `[a-z]`, `[^a-z]`在同一正则表达式中多次出现时, 不同位置的匹配是相互独立的, \
    例如`[ab][ab]`可以匹配`aa`, `ab`, `ba`或`bb`
* `\w`, `\W`, `\d`, `\D`均可作为方括号中的单字符通配符

In [8]:
tp_regex = [
    r"a", r".", 
    r"[aeiou]", r"[^aeiou]", r"[p-t]", r"[^p-t]", 
    r"\w", r"\W", r"[^aeiou\W]"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
a                       Verba volant, scripta manent
.                       Verba volant, scripta manent
[aeiou]                 Verba volant, scripta manent
[^aeiou]                Verba volant, scripta manent
[p-t]                   Verba volant, scripta manent
[^p-t]                  Verba volant, scripta manent
\w                      Verba volant, scripta manent
\W                      Verba volant, scripta manent
[^aeiou\W]              Verba volant, scripta manent


#### `\w`, `\d`的匹配范围演示

In [9]:
#表示文本或者数目的字符
str_alnum = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isalnum()]
); 

In [10]:
#正则表达式\w语法可以匹配的字符
re_mtch_word = str().join(re.compile("\w").findall(char)); 

In [11]:
for prop in "issuperset", "__eq__", "issubset": 
    print("{rel:\x20<12s}{propos!s}".format(
        rel=prop, 
        propos=set(re_mtch_word).__getattribute__(prop)(set(str_alnum))
    ) )

issuperset  True
__eq__      False
issubset    False


In [12]:
#正则表达式\w语法可以匹配的字符, 比满足isalnum的字符范围多一个_ (半角下划线)
set(re_mtch_word).difference(set(str_alnum))

{'_'}

In [13]:
#可用于十进制数码的字符
str_dec = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isdecimal()]
); 

In [14]:
#正则表达式\d语法可以匹配的字符
re_mtch_dec = str().join(re.compile("\d").findall(char)); 

In [15]:
for prop in "issuperset", "__eq__", "issubset": 
    print("{rel:\x20<12s}{propos!s}".format(
        rel=prop, 
        propos=set(re_mtch_dec).__getattribute__(prop)(set(str_dec))
    ) )

issuperset  True
__eq__      True
issubset    True


### 多字符匹配语法
|模式|功能|备注|
|:-|:-:|:-|
|`a?`|采用**贪心策略**匹配零个或<br>一个字符`a`|含该语法的模式会在符合匹配<br>规则的前提下, 匹配**最长**的子串|
|`a+`|采用贪心策略匹配一个<br>或**连续多个**字符`a`||
|`a*`|采用贪心策略匹配零个, <br>一个或**连续多个**字符`a`||
|`a{m}`|采用贪心策略匹配**连续<br>`m`个**字符`a`||
|`a{m,n}`|采用贪心策略匹配**连续<br>多个**字符`a`, 最少`m`个, <br>最多`n`个|`m`缺省时为`0`, `n`缺省时<br>为无穷大, 但逗号不可省略|
|`a??`, <br>`a+?`, <br>`a*?`, <br>`a{m}?`, <br>`a{m,n}?`|采用**懒惰策略**匹配特定<br>数量的字符`a`|含该语法的模式会在符合匹配<br>规则的前提下, 匹配**最短**的子串|

* `a?`, `a+`, `a*`, `a{m}`均可视为`a{m,n}`的特例, 分别与`a{,1}`, `a{1,}`, `a{,}`, `a{m,m}`等效. 
* 上述任何语法在同一正则表达式中多次出现时, 不同位置的匹配是相互独立的, \
    例如`a?b?c`可以匹配`c`, `ac`, `bc`或`abc` 

In [16]:
tp_regex = [
    r"a\w?nt", r"a\w+nt", r"a\w*nt", 
    r"[^aeiou\W]{2}", r"[^aeiou\W]{2,}", r"[^aeiou\W]{,2}", 
    r"a.+a", r"a.+?a"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
a\w?nt                  Verba volant, scripta manent
a\w+nt                  Verba volant, scripta manent
a\w*nt                  Verba volant, scripta manent
[^aeiou\W]{2}           Verba volant, scripta manent
[^aeiou\W]{2,}          Verba volant, scripta manent
[^aeiou\W]{,2}          Verba volant, scripta manent
a.+a                    Verba volant, scripta manent
a.+?a                   Verba volant, scripta manent


### 字符组合语法
|模式|功能|备注|
|:-|:-:|:-|
|`(abc)`|字符`abc`顺次构成的子串作为整体, <br>构成一个**匿名编组**参与匹配||
|`(?P<id>ab)`|字符`ab`构成**实名编组**参与<br>匹配, 编组代号为`id`|`id`的内容需为合法的<br>python关键字标识符|
|`ab\|cd`|匹配`ab`或`cd`|匹配范围的界限是未经<br>转义的以下字符: <br>`\|`左侧的未成对`(` <br>或`\|`右侧的未成对`)`<br>或`\|`|
|`(?P=id)`|同一正则表达式中, 代号为`id`的<br>字符编组重新调用|`(?P=id)`必须位于<br>`(?P<id>ab)`之后, 中间<br>允许插入其他正则表达式<br>语法, 或者直接相邻; |

* 前述的[多字符匹配语法](#多字符匹配语法)也适用于编组, 匹配的范围是**将编组作为整体后**重复特定次数的子串, \
    例如`(ab){1,3}`可以匹配`ab`, `abab`, `ababab`
* **同一次匹配中**, 同一代号的编组在**定义位置`(?P<id>ab)`和所有调用位置`(?P=id)`**匹配的字符串内容**完全相同**, 可用于**取消通配符匹配的独立性**, \
    例如`(?P<same>[ab])(?P=same)`只能匹配`aa`和`bb`, 不能匹配`ab`和`ba`, \
    而`[ab][ab]`和`[ab]{2}`均可匹配`aa`, `ab`, `ba`或`bb`

In [17]:
tp_regex = [
    r"[^aeiou\W][aeiou]{2,}", r"([^aeiou\W][aeiou]){2,}", 
    r"an|en", r"an|en(t)",  r"an(|en)t",  r"(an|en)t",  r"(an(|e))nt", 
    r"[aeiou].*?[aeiou]", r"(?P<v>[aeiou]).*?(?P=v)"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
[^aeiou\W][aeiou]{2,}   Verba volant, scripta manent
([^aeiou\W][aeiou]){2,} Verba volant, scripta manent
an|en                   Verba volant, scripta manent
an|en(t)                Verba volant, scripta manent
an(|en)t                Verba volant, scripta manent
(an|en)t                Verba volant, scripta manent
(an(|e))nt              Verba volant, scripta manent
[aeiou].*?[aeiou]       Verba volant, scripta manent
(?P<v>[aeiou]).*?(?P=v) Verba volant, scripta manent
